In [6]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

df = pd.read_csv("f1_ml_ready_dataset.csv")

print("Dataset loaded successfully.")
print("Original dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

target_column = "positionOrder"

if target_column not in df.columns:
    raise ValueError(f"Target column '{target_column}' not found in dataset.")

df[target_column] = pd.to_numeric(df[target_column], errors="coerce")

df = df.dropna(subset=[target_column]).copy()

print("\nDataset shape after dropping missing target values:", df.shape)

X = df.drop(columns=[target_column])
y = df[target_column]

X = X.select_dtypes(include=[np.number])

print("\nFeature matrix shape before cleaning:", X.shape)
print("Target shape:", y.shape)

all_missing_cols = X.columns[X.isnull().all()].tolist()

print("\nColumns with all missing values:")
if len(all_missing_cols) > 0:
    print(all_missing_cols)
    X = X.drop(columns=all_missing_cols)
else:
    print("No fully missing columns found.")

print("\nFeature matrix shape after removing fully missing columns:", X.shape)

missing_values = X.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("\nColumns with missing values before imputation:")
if len(missing_values) > 0:
    print(missing_values)
else:
    print("No missing values found.")

imputer = SimpleImputer(strategy="median")

X_imputed = imputer.fit_transform(X)

X = pd.DataFrame(
    X_imputed,
    columns=X.columns,
    index=X.index
)

print("\nTotal missing values after imputation:", X.isnull().sum().sum())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining samples:", X_train.shape)
print("Testing samples:", X_test.shape)

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

gb_model.fit(X_train, y_train)

print("\nGradient Boosting model training completed.")

y_pred = gb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("R² Score:", r2)

results = pd.DataFrame({
    "Actual_Position": y_test.values,
    "Predicted_Position": y_pred
})

print("\nActual vs Predicted Positions:")
print(results.head(20))

results.to_csv("gradient_boosting_actual_vs_predicted.csv", index=False)

print("\nActual vs predicted results saved as gradient_boosting_actual_vs_predicted.csv")

feature_importance = pd.Series(
    gb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

feature_importance.to_csv("gradient_boosting_feature_importance.csv")

print("\nFeature importance saved as gradient_boosting_feature_importance.csv")

with open("f1_gradient_boosting_model.pkl", "wb") as f:
    pickle.dump(gb_model, f)

with open("f1_gradient_boosting_imputer.pkl", "wb") as f:
    pickle.dump(imputer, f)

with open("f1_gradient_boosting_features.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

with open("f1_gradient_boosting_dropped_columns.pkl", "wb") as f:
    pickle.dump(all_missing_cols, f)

print("\nSaved files:")
print("1. f1_gradient_boosting_model.pkl")
print("2. f1_gradient_boosting_imputer.pkl")
print("3. f1_gradient_boosting_features.pkl")
print("4. f1_gradient_boosting_dropped_columns.pkl")
print("5. gradient_boosting_actual_vs_predicted.csv")
print("6. gradient_boosting_feature_importance.csv")

print("\nAll processes completed successfully.")

Dataset loaded successfully.
Original dataset shape: (26759, 53)

Columns:
['raceId', 'driverId', 'constructorId', 'number_x', 'grid', 'milliseconds', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'number_y', 'driver_nationality', 'constructor_name', 'constructor_nationality', 'year', 'round', 'circuitId', 'race_name', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time', 'driver_age', 'finished_race', 'driver_experience', 'constructor_experience', 'driver_circuit_experience', 'driver_avg_finish_before_race', 'driver_avg_points_before_race', 'driver_points_last_3', 'driver_points_last_5', 'driver_finish_last_3', 'driver_finish_last_5', 'previous_race_finish', 'previous_race_points', 'previous_grid', 'constructor_avg_finish_before_race', 'constructor_avg_points_before_race', 'constructor_points_last_3', 'constructor_points_last_5', 'season_driver_points_before_race', 'season_constructor_points_before_race'

In [9]:
import pandas as pd
import numpy as np
import pickle
import warnings

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

df = pd.read_csv("f1_ml_ready_dataset.csv")

print("Dataset loaded successfully.")
print("Original dataset shape:", df.shape)

target_column = "positionOrder"

if target_column not in df.columns:
    raise ValueError(f"Target column '{target_column}' not found in dataset.")

df[target_column] = pd.to_numeric(df[target_column], errors="coerce")

df = df.dropna(subset=[target_column]).copy()

print("Dataset shape after dropping missing target values:", df.shape)

X = df.drop(columns=[target_column])
y = df[target_column]

X = X.select_dtypes(include=[np.number])

print("\nFeature matrix shape before cleaning:", X.shape)
print("Target shape:", y.shape)

fully_missing_cols = X.columns[X.isnull().all()].tolist()

print("\nFully missing columns:")
print(fully_missing_cols)

X = X.drop(columns=fully_missing_cols)

print("\nFeature matrix shape after removing fully missing columns:", X.shape)

print("\nMissing values before imputation:")
print(X.isnull().sum().sort_values(ascending=False).head(10))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining samples:", X_train.shape)
print("Testing samples:", X_test.shape)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingRegressor(random_state=42))
])

param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__subsample": [0.7, 0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    error_score="raise"
)

random_search.fit(X_train, y_train)

best_gb = random_search.best_estimator_

print("\nBest Parameters:")
print(random_search.best_params_)

y_pred = best_gb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nTuned Gradient Boosting Performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

trained_model = best_gb.named_steps["model"]

feature_importance = pd.Series(
    trained_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

feature_importance.to_csv("gradient_boosting_feature_importance.csv")

print("\nFeature importance saved as gradient_boosting_feature_importance.csv")

with open("f1_gradient_boosting_tuned.pkl", "wb") as f:
    pickle.dump(best_gb, f)

print("\nModel saved as f1_gradient_boosting_tuned.pkl")

Dataset loaded successfully.
Original dataset shape: (26759, 53)
Dataset shape after dropping missing target values: (26759, 53)

Feature matrix shape before cleaning: (26759, 52)
Target shape: (26759,)

Fully missing columns:
['driver_age']

Feature matrix shape after removing fully missing columns: (26759, 51)

Missing values before imputation:
raceId                           0
previous_race_finish             0
finished_race                    0
driver_experience                0
constructor_experience           0
driver_circuit_experience        0
driver_avg_finish_before_race    0
driver_avg_points_before_race    0
driver_points_last_3             0
driver_points_last_5             0
dtype: int64

Training samples: (21407, 51)
Testing samples: (5352, 51)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best Parameters:
{'model__subsample': 0.9, 'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_depth': 5, 'model__lea